# Phase 3 baseline: Colab pipeline

Runs top to bottom, no cell edits, on a **T4 GPU runtime** (Runtime > Change runtime type).

This machine has no GPU and a slow connection (~1.25 MB/s to the HF CDN, `results/throughput_laptop.json`), so everything below runs here instead. Git is the bridge: this notebook clones the **public** repo anonymously (no token needed), and at the end copies results back to Drive for a laptop to commit -- Colab never pushes (see HANDOFF.md "Workflow").

One deviation from the plan as originally written: PLAN.md sketches "download" and "normalize" as two separate steps. In this codebase they are not separable -- `scripts/download_data.py` calls the canonical decode path (`src/data/normalize.py`) inline on every row as it streams the parquet, and never writes a raw file to disk (Phase 2 design: "nothing raw is ever written"). So Cell 2 below does both at once; there is no standalone normalization pass to run afterward.

## Cell 1 -- mount Drive, clone, install, verify the bridge

Runs `scripts/bench_throughput.py` last, so a broken clone/install/GPU fails here in well under a minute rather than 10+ minutes into a download.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/anniketkumar/aigc-detect.git"

REPO_DIR = "/content/repo"
DRIVE_ROOT = "/content/drive/MyDrive/aigc"

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/features", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/results", exist_ok=True)

In [ ]:
# Public repo -- anonymous clone, no auth, no token.
!git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!pip install -q -r requirements.txt

In [ ]:
import torch

has_gpu = torch.cuda.is_available()
print("CUDA available:", has_gpu)
print("GPU:", torch.cuda.get_device_name(0) if has_gpu else "NONE")
assert has_gpu, (
    "No GPU visible. Runtime > Change runtime type > T4 GPU, then Runtime > "
    "Restart session and rerun from the top. A CPU runtime will make cell 4 "
    "(CLIP feature caching) take hours instead of minutes."
)

In [ ]:
# The bridge check. If this fails or reports laptop-like throughput, stop and
# fix the environment before spending time on cell 2's download.
!python -m scripts.bench_throughput --out results/throughput_colab.json

## Cell 2 -- download + normalize (one step, see the note above), build manifests

Uses the default per-source quotas in `src/data/sources.py`: ~18.2k images, 4 training generators + 3 held-out generators (MidJourney, Gemini, FLUX.1-dev) + 3 real sources. Images land in `/content` (ephemeral) -- nothing here goes to Drive, per the Workflow note (free tier is 15 GB and images get re-pulled every session anyway).

In [ ]:
!python -m scripts.download_data --out data/corpus

In [ ]:
!python -m src.data.manifest --ledger data/corpus/ledger.csv --out data/manifests
!python -m scripts.make_data_stats

## Gate -- the two non-negotiable guards, plus the rest of the suite

`tests/test_manifest.py::test_real_corpus_manifest_builds_and_holds_every_invariant` only *engages* once `data/corpus/ledger.csv` exists (HANDOFF.md's documented known gap, `tests/test_manifest.py:456`) -- which is exactly now, for the first time this session. This is the one place in the whole pipeline where the real WildFake/DALL·E blocklist is actually exercised against real data. **Do not skip this cell or continue past a failure.**

In [ ]:
import subprocess

result = subprocess.run(["python", "-m", "pytest", "-q"])
if result.returncode != 0:
    raise SystemExit(
        "pytest failed. Do not continue past this cell -- in particular this "
        "is the run where test_manifest.py's WildFake content-hash guard and "
        "the DALL\u00b7E source-registry denylist test actually see real data "
        "(HANDOFF.md 'Non-negotiable'). A failure here is a disqualification "
        "risk, not a flaky test to retry past."
    )
print("pytest green, including the real-corpus WildFake/DALL\u00b7E guards")

## Cell 4 -- cache frozen CLIP ViT-B/16 features to Drive

One pass per split. After this, every later head experiment (Phase 3 baseline, and any Phase 4+ head swap) reads three small `.npy`/`.json` files and needs no GPU.

In [ ]:
FEATURES_DIR = f"{DRIVE_ROOT}/features"

for split in ("train", "val", "test"):
    !python -m scripts.cache_features \
        --manifest data/manifests/{split}.csv \
        --out {FEATURES_DIR}/{split} \
        --device cuda --batch-size 256

## Cell 5 -- train the baseline head

Linear head, BCE, no augmentation -- the deliberate control (HANDOFF.md). Checkpoint goes straight to Drive, not to `/content`, because free Colab disconnects on idle and this is cheap to lose only once.

In [ ]:
CKPT = f"{DRIVE_ROOT}/checkpoints/baseline.pt"

!python -m src.train \
    --features-dir {FEATURES_DIR} \
    --train-split train --val-split val \
    --out {CKPT} \
    --epochs 50 --device cuda

## Cell 6 -- run the Phase 1 harness on the held-out test split

`--split test` includes both the transform grid *and* the three held-out generators (MidJourney, Gemini, FLUX.1-dev), since `src/data/manifest.py` routes them there in full.

In [ ]:
!python -m src.evaluate \
    --model clip_linear --ckpt {CKPT} \
    --split test --out results/baseline/ \
    --device cuda

In [ ]:
import json

summary = json.loads(open("results/baseline/summary.json").read())["summary"]
clean_auroc = summary["clean_auroc"]
print(f"clean AUROC: {clean_auroc:.4f}")

if clean_auroc > 0.99:
    print(
        "\n*** STOP AND FLAG THIS ***\n"
        "HANDOFF.md: clean AUROC above 0.99 is far more likely to be a "
        "surviving leak than a good model, given what the Phase 2 audit found "
        "in raw SID_Set. Do not treat this as a good result -- check the "
        "held-out generator breakdown and re-run the Phase 2 normalization "
        "audit (tests/test_normalization_audit.py) before trusting it."
    )
elif not (0.85 <= clean_auroc <= 0.95):
    print(
        f"note: clean AUROC {clean_auroc:.4f} is outside the expected "
        "0.85-0.95 band (HANDOFF.md). Not necessarily wrong -- just worth a "
        "second look before reporting it as the baseline."
    )

## Cell 7 -- copy results back to Drive (no git push from Colab)

A laptop pulls this folder from Drive and commits it -- Colab never pushes, which removes an auth failure mode deliberately (HANDOFF.md "Workflow").

In [ ]:
import shutil

shutil.copytree("results/baseline", f"{DRIVE_ROOT}/results/baseline", dirs_exist_ok=True)
shutil.copy("results/data_stats.md", f"{DRIVE_ROOT}/results/data_stats.md")
print(f"copied to {DRIVE_ROOT}/results/baseline and {DRIVE_ROOT}/results/data_stats.md")
print("On the laptop: pull these from Drive into results/baseline/ and results/data_stats.md, then git add + commit + push.")